# preprocessing.parsing.ms_office.markitdown.utils

> Shared MarkItDown conversion and embedded-image utilities for Linux and Windows.

In [ ]:
# |default_exp preprocessing.parsing.ms_office.markitdown.utils

In [ ]:
# | hide
from nbdev.showdoc import *

## Shared configuration

In [ ]:
# | export
import base64
import binascii
import asyncio
import json
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path, PurePosixPath
from tempfile import TemporaryDirectory
from zipfile import BadZipFile, ZipFile

from dotenv import load_dotenv
from lxml import etree
from tqdm.auto import tqdm

In [ ]:
# | export
OFFICE_EXTENSIONS = frozenset({
    ".doc", ".docx", ".odt",
    ".ppt", ".pptx", ".odp",
    ".xls", ".xlsx", ".xlsm", ".xlsb", ".ods",
    ".csv", ".tsv",
})

IMAGE_EXTENSION_BY_MIME = {
    "jpeg": ".jpg",
    "jpg": ".jpg",
    "png": ".png",
    "gif": ".gif",
    "webp": ".webp",
    "bmp": ".bmp",
    "tiff": ".tiff",
    "svg+xml": ".svg",
    "wmf": ".wmf",
    "x-wmf": ".wmf",
    "emf": ".emf",
    "x-emf": ".emf",
    "vnd.microsoft.icon": ".ico",
}

MARKDOWN_DATA_IMAGE_RE = re.compile(
    r"(?P<prefix>!\[(?P<alt>[^\]]*)\]\(\s*)data:image/"
    r"(?P<mime>[-\w.+]+);base64,(?P<data>[A-Za-z0-9+/=\s]+?)"
    r"(?P<suffix>\s*\))",
    flags=re.IGNORECASE,
)
HTML_DATA_IMAGE_RE = re.compile(
    r"(?P<prefix><img\b[^>]*?\bsrc\s*=\s*(?P<quote>[\"']))data:image/"
    r"(?P<mime>[-\w.+]+);base64,(?P<data>[A-Za-z0-9+/=\s]+?)"
    r"(?P<suffix>(?P=quote)[^>]*>)",
    flags=re.IGNORECASE,
)
DATA_IMAGE_RE = MARKDOWN_DATA_IMAGE_RE


def _find_project_root(start: Path | str | None = None) -> Path:
    """Find PROJ_ROOT from the environment or a parent pyproject.toml."""
    configured_root = os.getenv("PROJ_ROOT")
    if configured_root:
        project_root = Path(configured_root).expanduser().resolve()
        if not project_root.is_dir():
            raise FileNotFoundError(f"PROJ_ROOT is not a directory: {project_root}")
        return project_root

    current = Path(start or Path.cwd()).expanduser().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError(f"Could not find pyproject.toml above {current}")


PROJ_ROOT = _find_project_root()
ENV_FILE = PROJ_ROOT / ".env"
if not ENV_FILE.is_file():
    raise FileNotFoundError(f"Project environment file not found: {ENV_FILE}")
load_dotenv(ENV_FILE, override=False)
EXIFTOOL_BINARY = os.getenv("EXIFTOOL_BINARY", "exiftool")
DEFAULT_MAX_CONCURRENCY = max(
    1,
    int(os.getenv("MARKITDOWN_MAX_CONCURRENCY", "4")),
)


async def get_office_files_root() -> Path:
    """Read and validate OFFICE_FILES_ROOT from PROJ_ROOT/.env."""
    configured_root = os.getenv("OFFICE_FILES_ROOT")
    if not configured_root:
        raise RuntimeError(f"OFFICE_FILES_ROOT is not configured in {ENV_FILE}")

    root = Path(configured_root).expanduser()
    if not root.is_absolute():
        root = PROJ_ROOT / root
    root = root.resolve()
    if not root.is_dir():
        raise FileNotFoundError(f"OFFICE_FILES_ROOT is not a directory: {root}")
    return root


def _validate_max_concurrency(max_concurrency: int) -> int:
    if max_concurrency < 1:
        raise ValueError("max_concurrency must be at least 1")
    return max_concurrency


async def _run_concurrently(
    items: list,
    worker,
    *,
    max_concurrency: int,
    description: str,
    unit: str,
    show_progress: bool,
) -> list:
    """Run awaitable item workers concurrently while preserving input order."""
    semaphore = asyncio.Semaphore(_validate_max_concurrency(max_concurrency))
    progress = tqdm(
        total=len(items),
        desc=description,
        unit=unit,
        dynamic_ncols=True,
        disable=not show_progress,
    )

    async def run_one(item):
        async with semaphore:
            try:
                return await worker(item)
            finally:
                progress.update(1)

    try:
        return await asyncio.gather(*(run_one(item) for item in items))
    finally:
        progress.close()


async def _run_subprocess(command: list[str]) -> tuple[bytes, bytes]:
    """Run a subprocess without blocking, including in Windows notebooks."""
    try:
        process = await asyncio.create_subprocess_exec(
            *command,
            stdout=asyncio.subprocess.PIPE,
            stderr=asyncio.subprocess.PIPE,
        )
    except NotImplementedError:
        # Windows Jupyter kernels can use a SelectorEventLoop, whose
        # subprocess transport is unimplemented. Offload the blocking
        # fallback so concurrent conversions still overlap.
        completed = await asyncio.to_thread(
            subprocess.run,
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            check=False,
        )
        returncode = completed.returncode
        stdout, stderr = completed.stdout, completed.stderr
    else:
        stdout, stderr = await process.communicate()
        returncode = process.returncode

    if returncode:
        raise subprocess.CalledProcessError(
            returncode,
            command,
            output=stdout.decode("utf-8", errors="replace"),
            stderr=stderr.decode("utf-8", errors="replace"),
        )
    return stdout, stderr


## Office conversion

In [ ]:
# | export
_PPTX_NAMESPACES = {
    "a": "http://schemas.openxmlformats.org/drawingml/2006/main",
    "p": "http://schemas.openxmlformats.org/presentationml/2006/main",
    "r": "http://schemas.openxmlformats.org/officeDocument/2006/relationships",
}
_PPTX_EMBED_ATTRIBUTE = f"{{{_PPTX_NAMESPACES['r']}}}embed"


def _remove_unusable_pptx_pictures(slide_xml: bytes) -> tuple[bytes, int]:
    """Remove picture shapes that have no embedded image relationship."""
    document = etree.fromstring(slide_xml)
    removed = 0
    for picture in document.xpath(".//p:pic", namespaces=_PPTX_NAMESPACES):
        blip = picture.find(f".//{{{_PPTX_NAMESPACES['a']}}}blip")
        if blip is not None and blip.get(_PPTX_EMBED_ATTRIBUTE):
            continue
        parent = picture.getparent()
        if parent is not None:
            parent.remove(picture)
            removed += 1

    if removed == 0:
        return slide_xml, 0
    return (
        etree.tostring(
            document,
            encoding="UTF-8",
            xml_declaration=True,
            standalone=True,
        ),
        removed,
    )


def _resolve_exiftool(exiftool_binary: Path | str) -> str:
    """Resolve the ExifTool executable or raise an actionable error."""
    configured = os.fspath(exiftool_binary)
    resolved = shutil.which(configured)
    if resolved:
        return resolved

    candidate = Path(configured).expanduser()
    if candidate.is_file():
        return str(candidate.resolve())
    raise FileNotFoundError(
        "ExifTool executable not found. Install ExifTool or set "
        "EXIFTOOL_BINARY to its executable path."
    )


def _validate_jpeg_frame(frame_data: bytes, frame_number: int) -> None:
    """Reject missing or malformed ExifTool frame output."""
    if not frame_data.startswith(b"\xff\xd8"):
        raise ValueError(
            f"ExifTool did not return JPEG data for MPO frame {frame_number}"
        )


async def extract_mpo_frames(
    mpo_path: Path | str,
    output_dir: Path | str | None = None,
    *,
    output_stem: str | None = None,
    overwrite: bool = False,
    exiftool_binary: Path | str = EXIFTOOL_BINARY,
) -> list[Path]:
    """Extract every MPO frame to a separate JPEG without recompression."""
    source = Path(mpo_path).expanduser().resolve()
    if not source.is_file():
        raise FileNotFoundError(f"MPO image not found: {source}")

    stem = output_stem or source.stem
    if not stem or Path(stem).name != stem:
        raise ValueError("output_stem must be a filename stem without directories")

    destination = (
        Path(output_dir).expanduser()
        if output_dir is not None
        else source.parent / f"{source.stem}_frames"
    )
    if not destination.is_absolute():
        destination = source.parent / destination
    destination = destination.resolve()

    exiftool = _resolve_exiftool(exiftool_binary)
    metadata_stdout, _ = await _run_subprocess(
        [
            exiftool,
            "-j",
            "-G1",
            "-a",
            "-s",
            "-NumberOfImages",
            "-MPImageStart",
            "-MPImageLength",
            str(source),
        ]
    )
    try:
        metadata = json.loads(metadata_stdout.decode("utf-8"))[0]
        frame_count = int(metadata["MPF0:NumberOfImages"])
        frame_ranges = [
            (
                int(metadata[f"MPImage{index}:MPImageStart"]),
                int(metadata[f"MPImage{index}:MPImageLength"]),
            )
            for index in range(1, frame_count + 1)
        ]
    except (IndexError, KeyError, TypeError, ValueError, json.JSONDecodeError) as error:
        raise ValueError(
            f"ExifTool did not identify {source} as a valid MPO image"
        ) from error
    if frame_count < 1:
        raise ValueError(f"MPO image contains no frames: {source}")

    source_data = await asyncio.to_thread(source.read_bytes)
    for frame_number, (frame_start, frame_length) in enumerate(frame_ranges, start=1):
        frame_end = frame_start + frame_length
        if frame_start < 0 or frame_length < 1 or frame_end > len(source_data):
            raise ValueError(f"Invalid byte range for MPO frame {frame_number}")

    width = max(2, len(str(frame_count)))
    output_paths = [
        destination / f"{stem}_frame_{index:0{width}d}.jpg"
        for index in range(1, frame_count + 1)
    ]
    existing = [path for path in output_paths if path.exists()]
    if existing and not overwrite:
        raise FileExistsError(f"MPO frame output already exists: {existing[0]}")

    with TemporaryDirectory(prefix="ribosome-mpo-") as temp_dir:
        temporary_root = Path(temp_dir)
        temporary_paths = [temporary_root / path.name for path in output_paths]
        await _run_subprocess(
            [
                exiftool,
                "-q",
                "-q",
                "-o",
                str(temporary_paths[0]),
                "-MPF:All=",
                "-Trailer:All=",
                str(source),
            ]
        )
        primary_data = await asyncio.to_thread(temporary_paths[0].read_bytes)
        _validate_jpeg_frame(primary_data, 1)

        for frame_number, temporary_path in enumerate(temporary_paths[1:], start=2):
            frame_start, frame_length = frame_ranges[frame_number - 1]
            frame_data = source_data[frame_start : frame_start + frame_length]
            _validate_jpeg_frame(frame_data, frame_number)
            await asyncio.to_thread(temporary_path.write_bytes, frame_data)

        await asyncio.to_thread(destination.mkdir, parents=True, exist_ok=True)
        await asyncio.gather(
            *(
                asyncio.to_thread(shutil.copyfile, temporary_path, output_path)
                for temporary_path, output_path in zip(
                    temporary_paths,
                    output_paths,
                    strict=True,
                )
            )
        )

    return output_paths


async def _extract_mpo_frame_bytes(
    image_data: bytes,
    *,
    exiftool_binary: Path | str = EXIFTOOL_BINARY,
) -> tuple[list[bytes], bool]:
    """Extract MPO bytes through ExifTool, leaving ordinary JPEG data unchanged."""
    if b"MPF\x00" not in image_data:
        return [image_data], False

    with TemporaryDirectory(prefix="ribosome-mpo-source-") as temp_dir:
        temporary_root = Path(temp_dir)
        source = temporary_root / "embedded.mpo"
        await asyncio.to_thread(source.write_bytes, image_data)
        frame_paths = await extract_mpo_frames(
            source,
            temporary_root / "frames",
            output_stem="embedded",
            overwrite=True,
            exiftool_binary=exiftool_binary,
        )
        return list(
            await asyncio.gather(
                *(asyncio.to_thread(path.read_bytes) for path in frame_paths)
            )
        ), True


async def _sanitize_pptx_for_markitdown(
    source: Path,
    target: Path,
    *,
    exiftool_binary: Path | str = EXIFTOOL_BINARY,
) -> tuple[dict[str, int], list[tuple[str, int, bytes]]]:
    """Write a temporary PPTX and preserve every embedded MPO frame."""
    repairs = {
        "removed_pictures": 0,
        "normalized_mpo_images": 0,
        "additional_mpo_frames": 0,
    }
    additional_frames: list[tuple[str, int, bytes]] = []
    with ZipFile(source, "r") as input_archive, ZipFile(target, "w") as output_archive:
        output_archive.comment = input_archive.comment
        for archive_entry in input_archive.infolist():
            payload = await asyncio.to_thread(input_archive.read, archive_entry)
            archive_path = archive_entry.filename.lower()
            if (
                archive_path.startswith("ppt/slides/slide")
                and archive_path.endswith(".xml")
            ):
                payload, removed = await asyncio.to_thread(
                    _remove_unusable_pptx_pictures,
                    payload,
                )
                repairs["removed_pictures"] += removed
            elif archive_path.startswith("ppt/media/") and archive_path.endswith(
                (".jpeg", ".jpg", ".jpe")
            ):
                frames, is_mpo = await _extract_mpo_frame_bytes(
                    payload,
                    exiftool_binary=exiftool_binary,
                )
                if is_mpo:
                    payload = frames[0]
                    repairs["normalized_mpo_images"] += 1
                    for frame_number, frame_data in enumerate(frames[1:], start=2):
                        additional_frames.append(
                            (archive_entry.filename, frame_number, frame_data)
                        )
                        repairs["additional_mpo_frames"] += 1
            await asyncio.to_thread(output_archive.writestr, archive_entry, payload)
    return repairs, additional_frames


async def _append_additional_mpo_frames(
    markdown_file: Path,
    additional_frames: list[tuple[str, int, bytes]],
) -> None:
    """Append preserved secondary MPO frames as extractable JPEG data URIs."""
    if not additional_frames:
        return

    appended = ["\n\n<!-- Additional MPO frames preserved by ExifTool. -->\n\n"]
    for archive_name, frame_number, frame_data in additional_frames:
        alt_text = f"{Path(archive_name).stem} MPO frame {frame_number}"
        encoded = base64.b64encode(frame_data).decode("ascii")
        appended.append(f"![{alt_text}](data:image/jpeg;base64,{encoded})\n\n")

    def append_text() -> None:
        with markdown_file.open("a", encoding="utf-8") as output:
            output.write("".join(appended))

    await asyncio.to_thread(append_text)


def _markitdown_command(source: Path, target: Path) -> list[str]:
    return [
        sys.executable,
        "-m",
        "markitdown",
        str(source),
        "-o",
        str(target),
        "--keep-data-uris",
    ]


async def _run_markitdown(source: Path, target: Path) -> None:
    await _run_subprocess(_markitdown_command(source, target))


def _conversion_error_message(error: Exception) -> str:
    if isinstance(error, subprocess.CalledProcessError) and error.stderr:
        stderr_lines = [line.strip() for line in error.stderr.splitlines() if line.strip()]
        if stderr_lines:
            return stderr_lines[-1]
    return str(error)


async def convert_office_to_md(
    root_folder: Path | str,
    output_root: Path | str | None = None,
    *,
    overwrite: bool = False,
    show_progress: bool = True,
    exiftool_binary: Path | str = EXIFTOOL_BINARY,
    max_concurrency: int = DEFAULT_MAX_CONCURRENCY,
) -> dict[str, object]:
    """Concurrently convert supported Office files to mirrored Markdown output."""
    root = Path(root_folder).expanduser().resolve()
    if not root.is_dir():
        raise FileNotFoundError(f"Office root is not a directory: {root}")

    md_root = Path(output_root).expanduser() if output_root else root / ".md"
    if not md_root.is_absolute():
        md_root = root / md_root
    md_root = md_root.resolve()
    await asyncio.to_thread(md_root.mkdir, parents=True, exist_ok=True)

    source_files = await asyncio.to_thread(
        lambda: sorted(
            path
            for path in root.rglob("*")
            if path.is_file()
            and path.suffix.lower() in OFFICE_EXTENSIONS
            and not path.name.startswith("~$")
        )
    )
    report: dict[str, object] = {
        "root": root,
        "output_root": md_root,
        "discovered": source_files,
        "converted": [],
        "skipped": [],
        "failed": [],
        "mpo_images_expanded": 0,
        "additional_mpo_frames": 0,
    }

    target_locks: dict[Path, asyncio.Lock] = {}
    for source in source_files:
        relative_source = source.relative_to(root)
        output_stem = source.stem.rstrip(" .") or "_"
        markdown_file = (
            md_root / relative_source.parent / output_stem / f"{output_stem}.md"
        )
        target_locks.setdefault(markdown_file, asyncio.Lock())

    async def convert_one(source: Path) -> tuple[str, object, dict[str, int]]:
        relative_source = source.relative_to(root)
        # Windows strips trailing spaces and periods from directory names.
        output_stem = source.stem.rstrip(" .") or "_"
        markdown_file = (
            md_root / relative_source.parent / output_stem / f"{output_stem}.md"
        )
        repairs = {
            "removed_pictures": 0,
            "normalized_mpo_images": 0,
            "additional_mpo_frames": 0,
        }

        async with target_locks[markdown_file]:
            if markdown_file.exists() and not overwrite:
                tqdm.write(f"Skipped existing: {markdown_file}")
                return "skipped", markdown_file, repairs

            await asyncio.to_thread(
                markdown_file.parent.mkdir,
                parents=True,
                exist_ok=True,
            )
            try:
                if source.suffix.lower() == ".pptx":
                    with TemporaryDirectory(prefix="ribosome-pptx-") as temp_dir:
                        sanitized_source = Path(temp_dir) / source.name
                        repairs, additional_frames = (
                            await _sanitize_pptx_for_markitdown(
                                source,
                                sanitized_source,
                                exiftool_binary=exiftool_binary,
                            )
                        )
                        conversion_source = (
                            sanitized_source if any(repairs.values()) else source
                        )
                        if any(repairs.values()):
                            tqdm.write(
                                f"Converting sanitized PPTX: {source} "
                                f"({repairs['removed_pictures']} unusable picture(s) "
                                f"removed, {repairs['normalized_mpo_images']} MPO "
                                f"image(s) expanded, "
                                f"{repairs['additional_mpo_frames']} additional "
                                "frame(s) preserved)"
                            )
                        await _run_markitdown(conversion_source, markdown_file)
                        await _append_additional_mpo_frames(
                            markdown_file,
                            additional_frames,
                        )
                else:
                    await _run_markitdown(source, markdown_file)
            except (
                BadZipFile,
                OSError,
                ValueError,
                etree.XMLSyntaxError,
                subprocess.CalledProcessError,
            ) as conversion_error:
                tqdm.write(
                    f"Failed: {source}: "
                    f"{_conversion_error_message(conversion_error)}"
                )
                return "failed", (source, conversion_error), repairs

            tqdm.write(f"Converted: {source} -> {markdown_file}")
            return "converted", markdown_file, repairs

    results = await _run_concurrently(
        source_files,
        convert_one,
        max_concurrency=max_concurrency,
        description="Converting Office files",
        unit="file",
        show_progress=show_progress,
    )
    for status, payload, repairs in results:
        report[status].append(payload)
        if status == "converted":
            report["mpo_images_expanded"] += repairs["normalized_mpo_images"]
            report["additional_mpo_frames"] += repairs["additional_mpo_frames"]

    return report


## Embedded-image extraction

In [ ]:
# | export
def _image_suffix(mime_subtype: str) -> str:
    """Return a safe filename extension for an image MIME subtype."""
    normalized = mime_subtype.lower()
    if normalized in IMAGE_EXTENSION_BY_MIME:
        return IMAGE_EXTENSION_BY_MIME[normalized]
    safe_subtype = re.sub(r"[^a-z0-9]+", "_", normalized).strip("_")
    return f".{safe_subtype or 'bin'}"


async def extract_base64_images(
    markdown_file_path: Path | str,
    image_output_folder: Path | str = "img",
) -> int:
    """Extract Markdown and HTML data-URI images into a relative folder."""
    markdown_file = Path(markdown_file_path).expanduser().resolve()
    if not markdown_file.is_file():
        raise FileNotFoundError(f"Markdown file not found: {markdown_file}")

    relative_image_dir = Path(image_output_folder)
    if relative_image_dir.is_absolute() or ".." in relative_image_dir.parts:
        raise ValueError("image_output_folder must stay inside the Markdown directory")

    image_dir = markdown_file.parent / relative_image_dir
    markdown_image_dir = PurePosixPath(relative_image_dir.as_posix())
    content = await asyncio.to_thread(markdown_file.read_text, encoding="utf-8")
    extracted_count = 0
    pending_images: list[tuple[Path, bytes]] = []

    def write_image(match: re.Match, alt_text: str) -> str | None:
        nonlocal extracted_count
        encoded = "".join(match.group("data").split())
        encoded += "=" * (-len(encoded) % 4)
        try:
            image_data = base64.b64decode(encoded, validate=True)
        except (ValueError, binascii.Error) as error:
            print(f"Invalid base64 image in {markdown_file}: {error}")
            return None

        extracted_count += 1
        safe_alt = re.sub(r"[^\w.-]+", "_", alt_text, flags=re.UNICODE).strip("._")
        safe_alt = safe_alt[:50] or "image"
        image_name = (
            f"{extracted_count:04d}_{safe_alt}{_image_suffix(match.group('mime'))}"
        )
        pending_images.append((image_dir / image_name, image_data))
        return (markdown_image_dir / image_name).as_posix()

    def replace_markdown_image(match: re.Match) -> str:
        image_link = write_image(match, match.group("alt"))
        if image_link is None:
            return match.group(0)
        return f'{match.group("prefix")}{image_link}{match.group("suffix").lstrip()}'

    def replace_html_image(match: re.Match) -> str:
        image_link = write_image(match, "image")
        if image_link is None:
            return match.group(0)
        return f'{match.group("prefix")}{image_link}{match.group("suffix")}'

    rewritten = MARKDOWN_DATA_IMAGE_RE.sub(replace_markdown_image, content)
    rewritten = HTML_DATA_IMAGE_RE.sub(replace_html_image, rewritten)
    if pending_images:
        await asyncio.to_thread(image_dir.mkdir, parents=True, exist_ok=True)
        await asyncio.gather(
            *(
                asyncio.to_thread(image_path.write_bytes, image_data)
                for image_path, image_data in pending_images
            )
        )
    if rewritten != content:
        await asyncio.to_thread(
            markdown_file.write_text,
            rewritten,
            encoding="utf-8",
        )
    return extracted_count


async def extract_base64_from_md(
    root_folder: Path | str,
    image_output_folder: Path | str = "img",
    *,
    show_progress: bool = True,
    max_concurrency: int = DEFAULT_MAX_CONCURRENCY,
) -> dict[str, object]:
    """Concurrently extract data-URI images from Markdown files below root."""
    root = Path(root_folder).expanduser().resolve()
    if not root.is_dir():
        raise FileNotFoundError(f"Markdown root is not a directory: {root}")

    report: dict[str, object] = {
        "processed": [],
        "failed": [],
        "image_count": 0,
        "images_extracted": 0,
    }
    markdown_files = await asyncio.to_thread(
        lambda: sorted(
            path for path in root.rglob("*.md") if path.is_file()
        )
    )
    async def extract_one(markdown_file: Path) -> tuple[str, object, int]:
        try:
            image_count = await extract_base64_images(
                markdown_file,
                image_output_folder,
            )
        except (OSError, ValueError) as error:
            tqdm.write(f"Failed to extract images from {markdown_file}: {error}")
            return "failed", (markdown_file, error), 0

        tqdm.write(f"Extracted {image_count} image(s): {markdown_file}")
        return "processed", markdown_file, image_count

    results = await _run_concurrently(
        markdown_files,
        extract_one,
        max_concurrency=max_concurrency,
        description="Extracting base64 images",
        unit="file",
        show_progress=show_progress,
    )
    for status, payload, image_count in results:
        report[status].append(payload)
        if status == "processed":
            report["image_count"] += image_count
            report["images_extracted"] += image_count

    return report


## Complete workflow

In [ ]:
# | export
async def process_office_files(
    root_folder: Path | str | None = None,
    *,
    output_root: Path | str | None = None,
    overwrite: bool = False,
    image_output_folder: Path | str = "img",
    show_progress: bool = True,
    exiftool_binary: Path | str = EXIFTOOL_BINARY,
    max_concurrency: int = DEFAULT_MAX_CONCURRENCY,
) -> dict[str, object]:
    """Convert Office files and extract their embedded data-URI images."""
    root = (
        await get_office_files_root()
        if root_folder is None
        else Path(root_folder)
    )
    root = root.expanduser().resolve()
    conversion = await convert_office_to_md(
        root,
        output_root,
        overwrite=overwrite,
        show_progress=show_progress,
        exiftool_binary=exiftool_binary,
        max_concurrency=max_concurrency,
    )
    extraction = await extract_base64_from_md(
        conversion["output_root"],
        image_output_folder=image_output_folder,
        show_progress=show_progress,
        max_concurrency=max_concurrency,
    )

    tqdm.write(
        "Finished: "
        f"{len(conversion['converted'])} converted, "
        f"{len(conversion['skipped'])} skipped, "
        f"{len(conversion['failed'])} conversion failure(s), "
        f"{extraction['image_count']} image(s) extracted, "
        f"{len(extraction['failed'])} extraction failure(s)."
    )
    return {
        "root": root,
        "output_root": conversion["output_root"],
        "conversion": conversion,
        "extraction": extraction,
    }


In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()